# Select Firms and Filings

| Item | Description |
|---|---|
| Goal | Fix the firm universe and the two filing samples that every later notebook of the topic-relevance rebuild reads. |
| Input | `sp500_10k_item1.parquet`, one row per 10-K filing with its Item 1 text and a usability flag, and `sample_firms.parquet`, the firm list with the S&P 500 flag; both under `/project2/mmiric_1435/acroh/llm_method/data`. |
| Output | `output/T3.4.1/filings.csv`: one row per kept filing with its sample flags. |
| Unit | A 10-K filing (accession number) of one firm (gvkey) for one fiscal year. |
| Sample | S&P 500-ever firms with a usable Item 1 in fiscal years 1999–2025. Two samples: all years, and fiscal years 2010, 2015, and 2020. |
| Rules | When a firm-year has two usable filings, keep the earlier-filed one, which is the ordinary full-year report; ties go to the lower accession number. The three-year sample keeps every firm with at least one of the three years and records how many it has. |
| Variables | • `in_all_years`: 1 for every kept filing, the all-years sample.<br>• `in_three_years`: 1 if the filing's fiscal year is 2010, 2015, or 2020.<br>• `n_three_years`: $T_i$, the number of the three fiscal years in which firm $i$ has a kept filing, from 1 to 3, repeated on each of the firm's rows; 0 for firms with none. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for how this file is used downstream.


## Step 1. Import packages

Read the saved tables and display checks.


In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display


## Step 2. Define file locations

Read the source files in place and save the output in `output/T3.4.1/` under the TEMP folder.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
SOURCE_DATA_FOLDER = Path("/project2/mmiric_1435/acroh/llm_method/data")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.4.1"  # the identifier names this notebook's output folder
ITEM1_FILE = SOURCE_DATA_FOLDER / "interim/repo_data/raw/03_sp500_10k_item1/sp500_10k_item1.parquet"
FIRMS_FILE = SOURCE_DATA_FOLDER / "raw/sec/sample_firms.parquet"
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output" / NOTEBOOK_NAME

YEAR_MIN = 1999
YEAR_MAX = 2025
THREE_YEARS = [2010, 2015, 2020]
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Outputs: {OUTPUT_FOLDER}")


Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.4.1


## Step 3. Load the filing table

Read every filing's identifiers, fiscal year, filing date, form type, and usability flag. The Item 1 text itself is not needed here.


In [3]:
filings = pd.read_parquet(
    ITEM1_FILE,
    columns=["accessionNo", "cik", "gvkey", "conm", "fyear", "periodEnd", "filedAt", "formType", "usable"],
)
assert filings["accessionNo"].is_unique
assert filings[["gvkey", "fyear", "filedAt"]].notna().all().all()
print(f"Filings in the source table: {len(filings):,}; firms: {filings['gvkey'].nunique():,}")
print(f"Form types: {filings['formType'].value_counts().to_dict()}")
display(filings.head(3))


Filings in the source table: 22,299; firms: 1,061
Form types: {'10-K': 20539, '10-K405': 1739, '10-KT': 21}


,accessionNo,cik,gvkey,conm,fyear,periodEnd,filedAt,formType,usable
0,0000912057-94-000081,61478,001013,ADC TELECOMMUNICATIONS INC,1993,1993-10-31,1994-01-12 05:00:00+00:00,10-K,True
1,0000912057-94-004277,61478,001013,ADC TELECOMMUNICATIONS INC,1994,1994-10-31,1994-12-22 05:00:00+00:00,10-K,True
2,0000061478-96-000002,61478,001013,ADC TELECOMMUNICATIONS INC,1995,1995-10-31,1996-01-05 05:00:00+00:00,10-K,True


## Step 4. Load the firm list

The firm list has one row per firm (gvkey) with its S&P 500 flag and SIC code. Only the flag is used here; the SIC code is attached later in the score assembly notebook.


In [4]:
firms = pd.read_parquet(FIRMS_FILE, columns=["gvkey", "conm", "sic", "in_sp500"])
assert firms["gvkey"].is_unique
print(f"Firms in the firm list: {len(firms):,}; flagged S&P 500-ever: {int(firms['in_sp500'].sum()):,}")
display(firms.head(3))


Firms in the firm list: 25,207; flagged S&P 500-ever: 1,075


,gvkey,conm,sic,in_sp500
0,007596,MOVIE SYSTEMS INC,4841,False
1,006314,K-TRON INTERNATIONAL INC,3823,False
2,011007,UC CORP,7385,False


## Step 5. Confirm the firm universe

Every firm in the filing table must appear in the firm list with the S&P 500 flag set. This confirms that the filing table is the S&P 500-ever universe and no extra membership filter is needed.


In [5]:
filing_firms = filings[["gvkey"]].drop_duplicates().merge(
    firms[["gvkey", "in_sp500"]], on="gvkey", how="left", validate="one_to_one"
)
assert filing_firms["in_sp500"].notna().all()
assert filing_firms["in_sp500"].all()
print(f"Firms in the filing table: {len(filing_firms):,}; all flagged S&P 500-ever.")


Firms in the filing table: 1,061; all flagged S&P 500-ever.


## Step 6. Keep usable filings in the sample window

Drop filings whose Item 1 could not be extracted (`usable` is false) and filings outside fiscal years 1999–2025. Report how many each rule removes.


In [6]:
n_source = len(filings)
not_usable = ~filings["usable"]
outside_window = ~filings["fyear"].between(YEAR_MIN, YEAR_MAX)
filings = filings.loc[~not_usable & ~outside_window].copy()
print(f"Dropped as not usable: {int(not_usable.sum()):,}")
print(f"Dropped as outside {YEAR_MIN}–{YEAR_MAX} (among usable): {int((outside_window & ~not_usable).sum()):,}")
print(f"Kept: {len(filings):,} of {n_source:,} filings; firms: {filings['gvkey'].nunique():,}")
display(filings.head(3))


Dropped as not usable: 333
Dropped as outside 1999–2025 (among usable): 3,164
Kept: 18,802 of 22,299 filings; firms: 1,058


,accessionNo,cik,gvkey,conm,fyear,periodEnd,filedAt,formType,usable
6,0000912057-99-011120,61478,001013,ADC TELECOMMUNICATIONS INC,1999,1999-10-31,1999-12-30 05:00:00+00:00,10-K,True
7,0000912057-01-000926,61478,001013,ADC TELECOMMUNICATIONS INC,2000,2000-10-31,2001-01-10 05:00:00+00:00,10-K405,True
8,0000912057-02-001614,61478,001013,ADC TELECOMMUNICATIONS INC,2001,2001-10-31,2002-01-16 05:00:00+00:00,10-K,True


## Step 7. Resolve firm-years with two usable filings

A few firm-years carry two usable filings. Show every pair with its period end date. They are of three kinds: a full-year 10-K followed by a transition-period report after a change of fiscal year end (10-KT, or a 10-K covering a few months); a full-year 10-K followed by the next year's report carrying the same fiscal-year label; and one same-day duplicate copy. In all three the earlier-filed filing is the ordinary full-year report for that label, so the rule keeps the earlier-filed filing, with the lower accession number as the tie-break.


In [7]:
duplicated_firm_years = filings.loc[filings.duplicated(["gvkey", "fyear"], keep=False)]
duplicated_firm_years = duplicated_firm_years.sort_values(["gvkey", "fyear", "filedAt"])
print(f"Firm-years with two usable filings: {duplicated_firm_years.groupby(['gvkey', 'fyear']).ngroups:,}")
display(duplicated_firm_years[["gvkey", "conm", "fyear", "formType", "periodEnd", "filedAt", "accessionNo"]])

filings = filings.sort_values(["gvkey", "fyear", "filedAt", "accessionNo"])
filings = filings.drop_duplicates(["gvkey", "fyear"], keep="first").copy()
assert not filings.duplicated(["gvkey", "fyear"]).any()
print(f"Filings after the rule: {len(filings):,}; firms: {filings['gvkey'].nunique():,}")


Firm-years with two usable filings: 23


,gvkey,conm,fyear,formType,periodEnd,filedAt,accessionNo
863,001722,ARCHER-DANIELS-MIDLAND CO,2012,10-K,2012-06-30,2012-08-25 01:11:10+00:00,0000007084-12-000034
864,001722,ARCHER-DANIELS-MIDLAND CO,2012,10-KT,2012-12-31,2013-02-28 22:24:04+00:00,0000007084-13-000014
4148,004737,FIRST HORIZON CORP,2000,10-K,2000-12-31,2001-03-28 05:00:00+00:00,0000950144-01-004077
4147,004737,FIRST HORIZON CORP,2000,10-K405,2001-03-31,2002-03-25 05:00:00+00:00,0000950144-02-002671
4862,005492,L3HARRIS TECHNOLOGIES INC,2019,10-K,2019-06-28,2019-08-22 19:04:00+00:00,0000202058-19-000090
4861,005492,L3HARRIS TECHNOLOGIES INC,2019,10-KT,2020-01-03,2020-03-03 21:17:25+00:00,0000202058-20-000011
6102,006669,LENNAR CORP,2004,10-K,2004-11-30,2005-02-12 02:32:09+00:00,0001193125-05-027428
6103,006669,LENNAR CORP,2004,10-K,2005-01-30,2006-02-09 22:34:38+00:00,0001193125-06-025438
6413,006821,LOUISIANA-PACIFIC CORP,2007,10-K,2007-12-31,2008-03-07 15:55:40+00:00,0001047469-08-002334
6414,006821,LOUISIANA-PACIFIC CORP,2007,10-K,2008-01-31,2009-02-27 22:28:41+00:00,0001193125-09-041236


Filings after the rule: 18,779; firms: 1,058


## Step 8. Flag the two samples

The all-years sample is every kept filing. The three-year sample is the filings for fiscal years 2010, 2015, and 2020; a firm stays in it with however many of the three years it has, and `n_three_years` records that count.


In [8]:
filings["in_all_years"] = 1
filings["in_three_years"] = filings["fyear"].isin(THREE_YEARS).astype(int)
three_year_counts = filings.loc[filings["in_three_years"].eq(1)].groupby("gvkey")["fyear"].nunique()
filings["n_three_years"] = filings["gvkey"].map(three_year_counts).fillna(0).astype(int)

firms_all_years = filings["gvkey"].nunique()
firms_three_years = int(three_year_counts.size)
print(f"All-years sample: {len(filings):,} filings, {firms_all_years:,} firms")
print(f"Three-year sample: {int(filings['in_three_years'].sum()):,} filings, {firms_three_years:,} firms")
print("Firms by number of the three years present:")
print(three_year_counts.value_counts().sort_index().rename("firms").to_string())
display(filings.head(3))


All-years sample: 18,779 filings, 1,058 firms
Three-year sample: 2,072 filings, 841 firms
Firms by number of the three years present:
fyear
1    153
2    145
3    543


,accessionNo,cik,gvkey,conm,fyear,periodEnd,filedAt,formType,usable,in_all_years,in_three_years,n_three_years
6,0000912057-99-011120,61478,001013,ADC TELECOMMUNICATIONS INC,1999,1999-10-31,1999-12-30 05:00:00+00:00,10-K,True,1,0,1
7,0000912057-01-000926,61478,001013,ADC TELECOMMUNICATIONS INC,2000,2000-10-31,2001-01-10 05:00:00+00:00,10-K405,True,1,0,1
8,0000912057-02-001614,61478,001013,ADC TELECOMMUNICATIONS INC,2001,2001-10-31,2002-01-16 05:00:00+00:00,10-K,True,1,0,1


## Step 9. Count filings and firms by fiscal year

A coverage table for the record: filings and firms per fiscal year in the all-years sample.


In [9]:
coverage = filings.groupby("fyear").agg(filings=("accessionNo", "size"), firms=("gvkey", "nunique"))
assert coverage["filings"].sum() == len(filings)
print(f"Fiscal years covered: {coverage.index.min()}–{coverage.index.max()}")
display(coverage.T)


Fiscal years covered: 1999–2025


fyear,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,...,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
filings,703,692,713,726,730,738,732,728,737,725,...,688,678,674,670,654,655,648,649,643,625
firms,703,692,713,726,730,738,732,728,737,725,...,688,678,674,670,654,655,648,649,643,625


## Step 10. Save the filing list

Save one row per kept filing with the sample flags. Later notebooks filter on `in_all_years` or `in_three_years` and never reapply the rules above.


In [10]:
output_columns = [
    "accessionNo", "gvkey", "cik", "conm", "fyear", "filedAt",
    "in_all_years", "in_three_years", "n_three_years",
]
filings_file = OUTPUT_FOLDER / "filings.csv"
filings[output_columns].to_csv(filings_file, index=False)
print(f"Saved: {filings_file}")
print(f"Rows: {len(filings):,}; columns: {output_columns}")


Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.4.1/filings.csv
Rows: 18,779; columns: ['accessionNo', 'gvkey', 'cik', 'conm', 'fyear', 'filedAt', 'in_all_years', 'in_three_years', 'n_three_years']
